# 生成侧提示路线对比

- 对应论文章节：第3.3.2节 生成侧补充对比实验
- 源脚本：`experiments/03_证伪实验/scripts/运行_生成侧提示路线对比.py`
- notebook 作用：直接查看代码与已保存结果，命令行运行仍以 `.py` 为准

这本 notebook 对应少样例风格锚定、自由式思维展开，以及基线直接作答三条生成侧提示路线。

## 命令行复现

```bash
cd /root/Velo
/root/Velo/.venv/bin/python experiments/03_证伪实验/scripts/运行_生成侧提示路线对比.py --variant baseline_prompt4
```

## 源码镜像

下面这一格保留 `.py` 的完整源码，主要用于现场查阅。

In [ ]:
"""比较少样例锚定、自由式思维展开等生成侧提示路线。"""

from __future__ import annotations

import argparse
import json
import math
import re
import sys
import time
from pathlib import Path
from typing import Any, Sequence

ROOT = Path(__file__).resolve().parents[1]
EXPERIMENTS_ROOT = ROOT.parent
IMPL_ROOT = EXPERIMENTS_ROOT / "04_算法实现"
if str(IMPL_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPL_ROOT))

from retrieval_pipeline.adaptive_evidence import _sentence_fact_candidates
from retrieval_pipeline.common import (
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_LLM_MODEL,
    NO_CONTEXT_ANSWER,
    clean_text,
    contains_refusal,
    ensure_dir,
    generate_answer,
    generate_json_payload,
    normalize_text,
    request_completion,
    rerank_documents,
    tokenize_text,
)
from retrieval_pipeline.datasets import load_crud_cases
from retrieval_pipeline.metrics import evaluate_crud_results
from retrieval_pipeline.pipeline import PipelineRunResult, PipelineVariant, RagExperimentPipeline

OUTPUT_ROOT = ROOT / "results" / "02_生成侧补充对比"

COMPLEX_CASE_IDS = (
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_008",
    "questanswer_3docs_003",
    "questanswer_3docs_004",
    "questanswer_3docs_006",
)

BASELINE_VARIANT = PipelineVariant(
    key="baseline_rrf_rerank_direct",
    label="基线直接作答",
    use_rerank=True,
    answer_prompt_style="simple",
    final_source_count=3,
    complex_source_count=5,
)

VARIANT_LABELS = {
    "baseline_prompt4": "基线直接作答",
    "fewshot_icl_direct": "少样例风格锚定",
    "scratchpad_direct": "自由式思维展开",
}

VARIANT_FILE_STEMS = {
    "baseline_prompt4": "生成侧_基线直接作答",
    "fewshot_icl_direct": "生成侧_少样例风格锚定",
    "scratchpad_direct": "生成侧_自由式思维展开",
}


def format_evidence_units(evidence_units: Sequence[dict[str, Any]]) -> str:
    blocks = []
    for index, unit in enumerate(evidence_units, start=1):
        blocks.append(f"[S{index}] {unit['title']}\n{unit['text']}")
    return "\n\n".join(blocks)


def extract_answer_tag(text: str) -> str:
    match = re.search(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        answer = clean_text(match.group(1), limit=600)
        return answer or NO_CONTEXT_ANSWER
    return clean_text(text, limit=600) or NO_CONTEXT_ANSWER


def parse_claim_strings(payload: dict[str, Any]) -> list[str]:
    raw_claims = payload.get("claims")
    if not isinstance(raw_claims, list):
        return []
    claims: list[str] = []
    for item in raw_claims:
        if isinstance(item, str):
            text = clean_text(item, limit=220)
        elif isinstance(item, dict):
            text = clean_text(str(item.get("text") or item.get("claim") or ""), limit=220)
        else:
            text = ""
        if text and normalize_text(text) != normalize_text(NO_CONTEXT_ANSWER):
            claims.append(text)
    return claims


def extract_numeric_tokens(text: str) -> set[str]:
    return set(re.findall(r"\d+(?:\.\d+)?", text))


def best_support_score(
    pipeline: RagExperimentPipeline,
    statement: str,
    candidates: Sequence[str],
) -> tuple[float, str]:
    if not candidates:
        return 0.0, ""
    raw_scores = pipeline.reranker.predict(
        [(statement, candidate) for candidate in candidates],
        batch_size=16,
        show_progress_bar=False,
    ).tolist()
    best_index = max(range(len(candidates)), key=lambda index: float(raw_scores[index]))
    best_score = 1.0 / (1.0 + math.exp(-float(raw_scores[best_index])))
    return best_score, candidates[best_index]


def build_verified_facts_prompt(query: str, facts: Sequence[str]) -> str:
    bullet_lines = [f"- {fact}" for fact in facts]
    return (
        "你是一个严格依据事实作答的问答助手。请仅根据下面已经核验过的事实回答问题。\n"
        "- 第一句必须直接回答问题。\n"
        "- 保留关键数值、日期、名称、条件和并列关系。\n"
        "- 如果问题需要多个事实点，请用自然语言连贯组织，但不要扩写背景。\n"
        f"- 如果事实仍不足，请原样回答：{NO_CONTEXT_ANSWER}\n\n"
        f"已核验事实：\n{chr(10).join(bullet_lines)}\n\n"
        f"用户问题：{query}\n"
        "回答："
    )


def build_fewshot_prompt(query: str, evidence_units: Sequence[dict[str, Any]]) -> str:
    context = format_evidence_units(evidence_units)
    return (
        "你是一个多文档问答助手。请模仿下面优秀回答的整合方式：先直接回答，再把不同文档中的关键细节缝合成自然语言，不要分点，不要补背景。\n\n"
        "[样例1]\n"
        "文档A：某企业2023年营业收入同比增长12%，净利润同比增长8%。\n"
        "文档B：该企业2023年研发投入同比增长15%，并推出三项核心产品。\n"
        "问题：2023年这家企业的经营表现和研发进展如何？\n"
        "优秀回答：2023年这家企业营业收入同比增长12%，净利润同比增长8%；同时研发投入同比增长15%，并推出了三项核心产品。\n\n"
        "[样例2]\n"
        "文档A：A地遭遇强降雨后启动Ⅲ级应急响应。\n"
        "文档B：B地转移群众1200人，并关闭沿河景区。\n"
        "问题：两地分别采取了哪些防灾措施？\n"
        "优秀回答：针对强降雨，A地启动了Ⅲ级应急响应；B地则转移了1200名群众，并关闭了沿河景区。\n\n"
        "[当前任务]\n"
        f"证据单元：\n{context}\n\n"
        f"问题：{query}\n"
        "优秀回答："
    )


def build_scratchpad_prompt(query: str, evidence_units: Sequence[dict[str, Any]]) -> str:
    context = format_evidence_units(evidence_units)
    return (
        "请根据以下文档回答问题。\n"
        f"文档：\n{context}\n\n"
        f"问题：{query}\n\n"
        "在给出最终答案前，请在 <thinking> 和 </thinking> 标签之间，用自然语言自由推演这些文档之间的关系、数字和线索。\n"
        f"如果证据不足，请在 <answer> 中原样输出：{NO_CONTEXT_ANSWER}\n"
        "推演结束后，在 <answer> 标签内输出完整、连贯的最终回答。\n"
    )


def build_citation_claim_prompt(query: str, evidence_units: Sequence[dict[str, Any]]) -> str:
    context = format_evidence_units(evidence_units)
    return (
        "你是一个带引用的答案起草器。请仅根据证据回答问题，并只输出 JSON。\n"
        "输出格式严格为："
        '{"claims":[{"text":"一个关键陈述","source_id":"S1","quote":"证据中的原句"}]}\n'
        "要求：\n"
        "1. 每条 claim 只包含一个关键陈述。\n"
        "2. source_id 只能填写对应证据编号。\n"
        "3. quote 必须是证据中的原句或连续片段。\n"
        "4. 不要输出背景概述，不要编造引用。\n\n"
        f"证据单元：\n{context}\n\n"
        f"问题：{query}\n"
        "JSON："
    )


def parse_citation_claims(payload: dict[str, Any], evidence_units: Sequence[dict[str, Any]]) -> list[str]:
    raw_claims = payload.get("claims")
    if not isinstance(raw_claims, list):
        return []
    source_map = {f"S{index}": str(unit["text"]) for index, unit in enumerate(evidence_units, start=1)}
    verified: list[str] = []
    for item in raw_claims:
        if not isinstance(item, dict):
            continue
        claim = clean_text(str(item.get("text") or ""), limit=220)
        source_id = clean_text(str(item.get("source_id") or ""), limit=8)
        quote = clean_text(str(item.get("quote") or ""), limit=220)
        source_text = source_map.get(source_id, "")
        if not claim or not quote or not source_text:
            continue
        if quote not in source_text:
            continue
        claim_numbers = extract_numeric_tokens(claim)
        quote_numbers = extract_numeric_tokens(quote)
        if claim_numbers and not claim_numbers.issubset(quote_numbers):
            verified.append(quote)
            continue
        verified.append(claim)
    return verified


def build_decompose_prompt(answer: str) -> str:
    return (
        "请把下面答案拆成原子事实列表，并只输出 JSON。\n"
        '格式严格为 {"claims":["事实1","事实2"]}\n'
        "要求：每条 claim 只包含一个明确陈述，不要重复。\n\n"
        f"答案：{answer}\n"
        "JSON："
    )


def mine_additional_facts(
    pipeline: RagExperimentPipeline,
    query: str,
    query_tokens: Sequence[str],
    evidence_units: Sequence[dict[str, Any]],
    covered: set[str],
    *,
    limit: int = 2,
) -> list[str]:
    facts = _sentence_fact_candidates(query_tokens, evidence_units, max_candidates_per_unit=2)
    candidates = [fact.statement for fact in facts if normalize_text(fact.statement) not in covered]
    if not candidates:
        return []
    raw_scores = pipeline.reranker.predict(
        [(query, candidate) for candidate in candidates],
        batch_size=16,
        show_progress_bar=False,
    ).tolist()
    ranked = sorted(
        zip(candidates, raw_scores),
        key=lambda item: float(item[1]),
        reverse=True,
    )
    selected: list[str] = []
    for candidate, score in ranked:
        if len(selected) >= limit:
            break
        if 1.0 / (1.0 + math.exp(-float(score))) < 0.62:
            continue
        selected.append(candidate)
    return selected


def build_baseline_evidence(
    pipeline: RagExperimentPipeline,
    prepared,
    case,
):
    features = prepared.query_features[case.case_id]
    candidate_indices, _ = pipeline._select_retrieval_view(prepared, features, BASELINE_VARIANT)
    reranked_doc_indices, doc_snippets, doc_scores = rerank_documents(
        pipeline.reranker,
        query=case.query,
        query_tokens=features.query_tokens,
        focus_tokens=features.query_tokens,
        candidate_doc_indices=candidate_indices,
        corpus_texts=[doc.content for doc in prepared.docs],
        multi_snippet_count=BASELINE_VARIANT.multi_snippet_count,
    )
    routing_units = pipeline._build_evidence_units(
        reranked_doc_indices=reranked_doc_indices,
        doc_snippets=doc_snippets,
        doc_scores=doc_scores,
        docs=prepared.docs,
        variant=BASELINE_VARIANT,
        source_limit=max(BASELINE_VARIANT.final_source_count, BASELINE_VARIANT.complex_source_count),
    )
    units = routing_units[: BASELINE_VARIANT.final_source_count]
    return features, units


def run_baseline_answer(pipeline: RagExperimentPipeline, query: str, evidence_units: Sequence[dict[str, Any]]) -> str:
    return pipeline._run_direct_answer(query, evidence_units, style=BASELINE_VARIANT.answer_prompt_style)


def run_fewshot_answer(query: str, evidence_units: Sequence[dict[str, Any]]) -> str:
    prompt = build_fewshot_prompt(query, evidence_units)
    return generate_answer(prompt, model_name=DEFAULT_LLM_MODEL)


def run_scratchpad_answer(query: str, evidence_units: Sequence[dict[str, Any]]) -> str:
    prompt = build_scratchpad_prompt(query, evidence_units)
    raw = request_completion(prompt, model_name=DEFAULT_LLM_MODEL, num_predict=640)
    return extract_answer_tag(raw)


def run_citation_revise_answer(
    pipeline: RagExperimentPipeline,
    query: str,
    evidence_units: Sequence[dict[str, Any]],
) -> str:
    payload = generate_json_payload(
        build_citation_claim_prompt(query, evidence_units),
        model_name=DEFAULT_LLM_MODEL,
        num_predict=480,
    )
    verified_claims = parse_citation_claims(payload, evidence_units)
    if not verified_claims:
        return run_baseline_answer(pipeline, query, evidence_units)
    prompt = build_verified_facts_prompt(query, verified_claims)
    return generate_answer(prompt, model_name=DEFAULT_LLM_MODEL)


def run_decompose_verify_answer(
    pipeline: RagExperimentPipeline,
    query: str,
    query_tokens: Sequence[str],
    evidence_units: Sequence[dict[str, Any]],
) -> str:
    baseline_answer = run_baseline_answer(pipeline, query, evidence_units)
    payload = generate_json_payload(
        build_decompose_prompt(baseline_answer),
        model_name=DEFAULT_LLM_MODEL,
        num_predict=320,
    )
    claims = parse_claim_strings(payload)
    if not claims:
        return baseline_answer

    fact_candidates = [fact.statement for fact in _sentence_fact_candidates(query_tokens, evidence_units, max_candidates_per_unit=3)]
    kept_facts: list[str] = []
    covered: set[str] = set()
    for claim in claims:
        score, support = best_support_score(pipeline, claim, fact_candidates)
        if score < 0.60 or not support:
            continue
        signature = normalize_text(support)
        if signature in covered:
            continue
        covered.add(signature)
        kept_facts.append(support)
    kept_facts.extend(mine_additional_facts(pipeline, query, query_tokens, evidence_units, covered))
    if not kept_facts:
        return baseline_answer
    prompt = build_verified_facts_prompt(query, kept_facts)
    return generate_answer(prompt, model_name=DEFAULT_LLM_MODEL)


def run_variant_case(
    pipeline: RagExperimentPipeline,
    prepared,
    case,
    variant_key: str,
) -> PipelineRunResult:
    started = time.perf_counter()
    features, evidence_units = build_baseline_evidence(pipeline, prepared, case)
    if variant_key == "baseline_prompt4":
        answer = run_baseline_answer(pipeline, case.query, evidence_units)
    elif variant_key == "fewshot_icl_direct":
        answer = run_fewshot_answer(case.query, evidence_units)
    elif variant_key == "scratchpad_direct":
        answer = run_scratchpad_answer(case.query, evidence_units)
    else:
        raise ValueError(f"Unknown variant: {variant_key}")

    latency_ms = (time.perf_counter() - started) * 1000.0
    source_doc_ids: list[int] = []
    source_titles: list[str] = []
    for unit in evidence_units:
        doc_id = int(unit["doc_id"])
        if doc_id in source_doc_ids:
            continue
        source_doc_ids.append(doc_id)
        source_titles.append(str(unit["title"]))
    return PipelineRunResult(
        case_id=case.case_id,
        query=case.query,
        response=answer or NO_CONTEXT_ANSWER,
        source_doc_ids=source_doc_ids,
        source_titles=source_titles,
        retrieved_contexts=[str(unit["text"]) for unit in evidence_units],
        latency_ms=latency_ms,
        predicted_refusal=contains_refusal(answer),
        route_mode=variant_key,
    )


def main() -> None:
    parser = argparse.ArgumentParser(description="在 CRUD 复杂问答批次上运行生成侧提示路线对比。")
    parser.add_argument(
        "--variant",
        choices=tuple(VARIANT_LABELS),
        required=True,
    )
    parser.add_argument("--output-suffix", default="")
    args = parser.parse_args()

    ensure_dir(OUTPUT_ROOT)
    cache_root = ensure_dir(ROOT / ".cache")

    cases, docs = load_crud_cases(
        summary_samples=6,
        qa_1doc_samples=6,
        qa_2doc_samples=8,
        qa_3doc_samples=8,
        hallu_samples=4,
        negative_samples=6,
        distractor_count=600,
        seed=42,
    )
    selected_cases = [case for case in cases if case.case_id in COMPLEX_CASE_IDS]

    pipeline = RagExperimentPipeline(
        cache_root=cache_root,
        embedding_model=DEFAULT_EMBEDDING_MODEL,
        llm_model=DEFAULT_LLM_MODEL,
    )
    prepared = pipeline.prepare_dataset(
        "crud_generation_prompt_compare_batch",
        selected_cases,
        docs,
        include_contextual=False,
        include_parent_child=False,
        include_query_rewrite=False,
    )

    results: list[PipelineRunResult] = []
    total = len(selected_cases)
    for index, case in enumerate(selected_cases, start=1):
        if index == 1 or index == total:
            print(f"[crud-prompt4] {args.variant}: {index}/{total}", flush=True)
        results.append(run_variant_case(pipeline, prepared, case, args.variant))

    evaluation = evaluate_crud_results(
        args.variant,
        results,
        selected_cases,
        ragas_case_ids=(),
        qa_ragas_case_ids=(),
        multidoc_ragas_case_ids=(),
        enable_ragas=False,
    )
    summary = dict(evaluation.summary)
    summary["label"] = VARIANT_LABELS[args.variant]
    payload = {"case_ids": list(COMPLEX_CASE_IDS), "summaries": [summary]}
    stem = VARIANT_FILE_STEMS[args.variant]
    suffix = f"_{args.output_suffix}" if args.output_suffix else ""
    (OUTPUT_ROOT / f"{stem}_结果汇总{suffix}.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    (OUTPUT_ROOT / f"{stem}_逐题明细{suffix}.json").write_text(
        json.dumps(evaluation.detail_rows, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(payload, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## 结果预览

下面直接内嵌当前已保存结果的关键文件预览。

### 生成侧：基线直接作答

- 文件：`../results/02_生成侧补充对比/生成侧_基线直接作答_结果汇总.json`

In [1]:
from pathlib import Path
import json

path = Path('../results/02_生成侧补充对比/生成侧_基线直接作答_结果汇总.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "case_ids": [
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_008",
    "questanswer_3docs_003",
    "questanswer_3docs_004",
    "questanswer_3docs_006"
  ],
  "summaries": [
    {
      "variant": "baseline_rrf_rerank_direct",
      "dataset": "crud",
      "faithfulness": 0.0,
      "answer_correctness": 0.0,
      "answer_relevancy": 0.0,
      "context_precision": 0.0,
      "ragas_sample_count": 0,
      "accuracy": 0.0,
      "qa_accuracy": 0.0,
      "retrieval_hit_rate_at_1": 0.875,
      "retrieval_hit_rate_at_3": 1.0,
      "qa_faithfulness": 0.0,
      "qa_answer_correctness": 0.0,
      "qa_answer_relevancy": 0.0,
      "qa_context_precision": 0.0,
      "qa_ragas_sample_count": 0,
      "multidoc_faithfulness": 0.0,
      "multidoc_answer_correctness": 0.0,
      "multidoc_answer_relevancy": 0.0,
      "multidoc_context_precision": 0.0,
      "multidoc_ragas_sample_count": 0,
  

### 生成侧：少样例风格锚定

- 文件：`../results/02_生成侧补充对比/生成侧_少样例风格锚定_结果汇总.json`

In [2]:
from pathlib import Path
import json

path = Path('../results/02_生成侧补充对比/生成侧_少样例风格锚定_结果汇总.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "case_ids": [
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_008",
    "questanswer_3docs_003",
    "questanswer_3docs_004",
    "questanswer_3docs_006"
  ],
  "summaries": [
    {
      "variant": "fewshot_icl_direct",
      "dataset": "crud",
      "faithfulness": 0.0,
      "answer_correctness": 0.0,
      "answer_relevancy": 0.0,
      "context_precision": 0.0,
      "ragas_sample_count": 0,
      "accuracy": 0.0,
      "qa_accuracy": 0.0,
      "retrieval_hit_rate_at_1": 0.875,
      "retrieval_hit_rate_at_3": 1.0,
      "qa_faithfulness": 0.0,
      "qa_answer_correctness": 0.0,
      "qa_answer_relevancy": 0.0,
      "qa_context_precision": 0.0,
      "qa_ragas_sample_count": 0,
      "multidoc_faithfulness": 0.0,
      "multidoc_answer_correctness": 0.0,
      "multidoc_answer_relevancy": 0.0,
      "multidoc_context_precision": 0.0,
      "multidoc_ragas_sample_count": 0,
      "ove

### 生成侧：自由式思维展开

- 文件：`../results/02_生成侧补充对比/生成侧_自由式思维展开_结果汇总.json`

In [3]:
from pathlib import Path
import json

path = Path('../results/02_生成侧补充对比/生成侧_自由式思维展开_结果汇总.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "case_ids": [
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_008",
    "questanswer_3docs_003",
    "questanswer_3docs_004",
    "questanswer_3docs_006"
  ],
  "summaries": [
    {
      "variant": "scratchpad_direct",
      "dataset": "crud",
      "faithfulness": 0.0,
      "answer_correctness": 0.0,
      "answer_relevancy": 0.0,
      "context_precision": 0.0,
      "ragas_sample_count": 0,
      "accuracy": 0.0,
      "qa_accuracy": 0.0,
      "retrieval_hit_rate_at_1": 0.875,
      "retrieval_hit_rate_at_3": 1.0,
      "qa_faithfulness": 0.0,
      "qa_answer_correctness": 0.0,
      "qa_answer_relevancy": 0.0,
      "qa_context_precision": 0.0,
      "qa_ragas_sample_count": 0,
      "multidoc_faithfulness": 0.0,
      "multidoc_answer_correctness": 0.0,
      "multidoc_answer_relevancy": 0.0,
      "multidoc_context_precision": 0.0,
      "multidoc_ragas_sample_count": 0,
      "over